# CDS 部署版：台风追踪与提取（历史业务预报）

本 Notebook 面向 CDS JupyterLab（4 CPU / 16GB），包含：依赖安装、数据请求、并行追踪、结果汇总。

稳定性策略：限并发、分批处理、失败重试、逐批落盘，避免卡死和内存崩溃。

In [ ]:
import os
import sys
import json
import time
import logging
import subprocess
from pathlib import Path

PROJECT_ROOT = Path('/home/jovyan/TianGong-AI-Cyclone') if Path('/home/jovyan/TianGong-AI-Cyclone').exists() else Path.cwd()
os.chdir(PROJECT_ROOT)

RAW_DIR = PROJECT_ROOT / 'data' / 'ecmwf_raw'
NC_DIR = PROJECT_ROOT / 'data' / 'ecmwf_nc'
TRACK_DIR = PROJECT_ROOT / 'track_output_cds'
SUMMARY_DIR = PROJECT_ROOT / 'output' / 'cds_notebook_summary'
for p in [RAW_DIR, NC_DIR, TRACK_DIR, SUMMARY_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CPU_COUNT = os.cpu_count() or 4
MAX_WORKERS = max(1, min(2, CPU_COUNT - 1))
CHUNK_SIZE = 4

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
print(f'PROJECT_ROOT={PROJECT_ROOT}')
print(f'MAX_WORKERS={MAX_WORKERS}, CHUNK_SIZE={CHUNK_SIZE}')

In [ ]:
def pip_install(packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', *packages]
    print('>>>', ' '.join(cmd))
    subprocess.check_call(cmd)

pip_install(['-r', 'requirements.txt'])
pip_install(['ecmwf-opendata', 'cfgrib', 'eccodes', 'psutil', 'tenacity', 's3fs'])
print('依赖安装完成')

## 数据请求配置

- `SOURCE=aws_open_data`：默认可直接跑。
- `SOURCE=tigge`：需你已有 ECMWF/TIGGE 授权与接口凭据。

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential

SOURCE = 'aws_open_data'
REQUEST_CONFIG = {
    'date': '2024-09-01',
    'run_hour': 0,
    'steps': list(range(0, 121, 6)),
    'params': ['msl', '10u', '10v', 'z', 'lsm'],
    'type': 'fc',
    'stream': 'oper'
}

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=2, min=2, max=30))
def request_aws_open_data(cfg, out_dir: Path):
    from ecmwf.opendata import Client
    out_dir.mkdir(parents=True, exist_ok=True)
    target = out_dir / f"ecmwf_{cfg['date'].replace('-', '')}_{int(cfg['run_hour']):02d}.grib2"
    client = Client(source='aws', preserve_request_order=True)
    client.retrieve(
        date=int(cfg['date'].replace('-', '')),
        time=int(cfg['run_hour']),
        step=cfg['steps'],
        stream=cfg['stream'],
        type=cfg['type'],
        param=cfg['params'],
        target=str(target),
    )
    return target

def request_tigge_placeholder(cfg, out_dir: Path):
    raise RuntimeError('请将此函数替换为你已授权的 TIGGE 请求实现（MARS/新接口）。')

raw_file = request_aws_open_data(REQUEST_CONFIG, RAW_DIR) if SOURCE == 'aws_open_data' else request_tigge_placeholder(REQUEST_CONFIG, RAW_DIR)
print('下载完成:', raw_file)

In [ ]:
import xarray as xr

def grib_to_netcdf(grib_path: Path, nc_dir: Path) -> Path:
    nc_dir.mkdir(parents=True, exist_ok=True)
    out_nc = nc_dir / (grib_path.stem + '.nc')
    if out_nc.exists() and out_nc.stat().st_size > 0:
        return out_nc
    ds = xr.open_dataset(grib_path, engine='cfgrib')
    rename_map = {}
    if 'u10' in ds.data_vars: rename_map['u10'] = '10u'
    if 'v10' in ds.data_vars: rename_map['v10'] = '10v'
    if rename_map: ds = ds.rename(rename_map)
    ds.to_netcdf(out_nc)
    ds.close()
    return out_nc

nc_file = grib_to_netcdf(Path(raw_file), NC_DIR)
print('转换完成:', nc_file)

In [ ]:
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing import get_context

INITIALS_CSV = PROJECT_ROOT / 'input' / 'western_pacific_typhoons_superfast.csv'

def _track_one_file(nc_path: str, initials_csv: str, output_dir: str, max_storms=None):
    from pathlib import Path
    import pandas as pd
    from initial_tracker.workflow import track_file_with_initials
    all_points = pd.read_csv(initials_csv)
    written = track_file_with_initials(
        nc_path=Path(nc_path),
        all_points=all_points,
        output_dir=Path(output_dir),
        max_storms=max_storms,
        time_window_hours=6
    )
    return [str(p) for p in written]

def batched(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i+size]

nc_files = sorted(str(p) for p in NC_DIR.glob('*.nc'))
if not nc_files:
    raise RuntimeError('未找到可追踪的 nc 文件，请先完成数据请求与转换')

all_outputs = []
for batch_idx, batch in enumerate(batched(nc_files, CHUNK_SIZE), start=1):
    print(f'\n=== Batch {batch_idx}: {len(batch)} files ===')
    with ProcessPoolExecutor(max_workers=MAX_WORKERS, mp_context=get_context('spawn')) as ex:
        futs = [ex.submit(_track_one_file, f, str(INITIALS_CSV), str(TRACK_DIR), None) for f in batch]
        for fut in as_completed(futs):
            try:
                outs = fut.result(timeout=3600)
                all_outputs.extend(outs)
                print(f'完成: {len(outs)} 条轨迹')
            except Exception as e:
                print('子任务失败:', e)
    time.sleep(2)

print(f'追踪结束，共输出 {len(all_outputs)} 个轨迹文件到 {TRACK_DIR}')

In [ ]:
import pandas as pd

track_files = sorted(TRACK_DIR.glob('*.csv'))
if not track_files:
    raise RuntimeError('没有找到追踪输出，请检查前序步骤日志')

frames = []
for f in track_files:
    df = pd.read_csv(f)
    if not df.empty:
        df['track_file'] = f.name
        frames.append(df)

all_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
summary = {
    'track_files': len(track_files),
    'total_points': int(len(all_df)),
    'time_min': str(all_df['time'].min()) if 'time' in all_df.columns and not all_df.empty else None,
    'time_max': str(all_df['time'].max()) if 'time' in all_df.columns and not all_df.empty else None,
    'min_msl': float(all_df['msl'].min()) if 'msl' in all_df.columns and not all_df.empty else None,
    'max_wind': float(all_df['wind'].max()) if 'wind' in all_df.columns and not all_df.empty else None
}

summary_path = SUMMARY_DIR / 'tracking_summary.json'
with open(summary_path, 'w', encoding='utf-8') as fw:
    json.dump(summary, fw, ensure_ascii=False, indent=2)

display(pd.DataFrame([summary]))
print('摘要写入:', summary_path)

## CDS 4C16G 参数建议

- `MAX_WORKERS=2`（重任务不建议超过 3）
- `CHUNK_SIZE=4` 左右
- 子任务超时保留（示例 3600 秒）
- 每批后短暂停顿，减少 I/O 和内存抖动
- 先跑单日/单台风，再扩展批量